# LeetCode 3856 — Trim Trailing Vowels bằng Python

Link đề: https://leetcode.com/problems/trim-trailing-vowels/ · Độ khó: **Easy** · Chủ đề: String, Two Pointers, Simulation

Skeleton LeetCode:

```python
class Solution:
    def trimTrailingVowels(self, s: str) -> str:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xóa **toàn bộ nguyên âm liên tiếp ở cuối chuỗi** (nguyên âm là `a, e, i, o, u`), chỉ ở **cuối**, không đụng tới nguyên âm ở giữa hoặc đầu chuỗi.
2. Biết kiểm tra thuộc tập bằng `ch in "aeiou"` / `set`, và biết cắt chuỗi `s[:k]`.
3. Biết duyệt chuỗi **từ phải sang trái** bằng chỉ số `i = len(s) - 1` giảm dần.
4. Biết các công cụ có sẵn: `str.rstrip(chars)`, biểu thức chính quy `re.sub(r'[aeiou]+$', '', s)`, `itertools.dropwhile`.
5. Viết 5 cách khác nhau, test ngẫu nhiên có `random.seed`, và tránh các lỗi: xóa mọi nguyên âm, chỉ xóa một nguyên âm, nhầm `strip` hai đầu, nhầm `removesuffix` với `rstrip`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các chữ cái tiếng Anh viết thường. Hãy **xóa tất cả nguyên âm ở cuối chuỗi** rồi trả về chuỗi còn lại. Nguyên âm là các chữ `a`, `e`, `i`, `o`, `u`. "Nguyên âm ở cuối" nghĩa là một **đoạn liên tiếp** các nguyên âm kết thúc tại ký tự cuối cùng của `s`. Nếu mọi ký tự đều là nguyên âm, kết quả là chuỗi rỗng.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"idea"` | `"id"` | Hai nguyên âm cuối `"ea"` bị xóa, còn `"id"` (nguyên âm `i` ở đầu giữ nguyên) |
| 2 | `"day"` | `"day"` | Ký tự cuối `y` là phụ âm (trong bài này `y` không phải nguyên âm) nên không xóa gì |
| 3 | `"aeiou"` | `""` | Mọi ký tự đều là nguyên âm |

Constraints: `1 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh viết thường.

In [1]:
for s in ["idea", "day", "aeiou", "banana", "rhythm"]:
    print(f"{s!r:10} ký tự cuối = {s[-1]!r}, là nguyên âm? {s[-1] in 'aeiou'}")

'idea'     ký tự cuối = 'a', là nguyên âm? True
'day'      ký tự cuối = 'y', là nguyên âm? False
'aeiou'    ký tự cuối = 'u', là nguyên âm? True
'banana'   ký tự cuối = 'a', là nguyên âm? True
'rhythm'   ký tự cuối = 'm', là nguyên âm? False


## 3. Kiến thức nền

### 3.1. Truy cập ký tự theo chỉ số, chỉ số âm

`s[0]` là ký tự đầu, `s[-1]` là ký tự cuối, `s[-2]` là ký tự gần cuối, ... Độ dài `len(s)`; chỉ số hợp lệ từ `0` đến `len(s) - 1`.

In [2]:
s = "idea"
print(s[0], s[3], s[-1], s[-2], len(s))

i a a e 4


### 3.2. Kiểm tra thuộc tập: `in`

`ch in "aeiou"` là `True` nếu `ch` là một trong năm chữ. Với `set("aeiou")` việc tra cứu là `O(1)`, với chuỗi ngắn như vậy khác biệt không đáng kể.

In [3]:
vowels = set("aeiou")
print('a' in vowels, 'y' in vowels, 'a' in "aeiou", 'y' in "aeiou")

True False True False


### 3.3. Cắt lát (slicing) `s[:k]`

`s[:k]` lấy `k` ký tự đầu. `s[:0]` là chuỗi rỗng. Nếu `k` bằng độ dài thì ta được chính chuỗi. Vì vậy sau khi tìm được vị trí kết thúc `end` (số ký tự giữ lại), kết quả là `s[:end]`.

In [4]:
s = "idea"
print(repr(s[:0]), repr(s[:2]), repr(s[:4]), repr(s[:10]))

'' 'id' 'idea' 'idea'


### 3.4. `str.rstrip(chars)`

`s.rstrip("aeiou")` xóa **từ cuối** mọi ký tự thuộc tập `{a, e, i, o, u}` cho đến khi gặp ký tự không thuộc tập. Tham số là **tập ký tự**, không phải một chuỗi con. Tương tự `lstrip` xóa ở đầu, `strip` xóa ở cả hai đầu.

In [5]:
print(repr("idea".rstrip("aeiou")), repr("day".rstrip("aeiou")), repr("aeiou".rstrip("aeiou")))
print(repr("ouch".lstrip("aeiou")), repr("ouchu".strip("aeiou")))

'id' 'day' ''
'ch' 'ch'


### 3.5. Biểu thức chính quy `re.sub`

`re.sub(mẫu, thay_thế, s)` thay các đoạn khớp mẫu. Mẫu `[aeiou]+$` nghĩa: một hoặc nhiều nguyên âm (`[aeiou]+`) nằm sát cuối chuỗi (`$`). Thay bằng chuỗi rỗng là xóa đoạn đó.

In [6]:
import re
print(repr(re.sub(r"[aeiou]+$", "", "idea")), repr(re.sub(r"[aeiou]+$", "", "banana")), repr(re.sub(r"[aeiou]+$", "", "aeiou")))

'id' 'banan' ''


### 3.6. `itertools.dropwhile`

`dropwhile(điều_kiện, iterable)` bỏ các phần tử **đầu** cho đến khi điều kiện sai lần đầu, giữ phần còn lại. Áp dụng cho `reversed(s)` ta bỏ nguyên âm ở cuối chuỗi gốc.

In [7]:
from itertools import dropwhile
print(list(dropwhile(lambda c: c in "aeiou", reversed("idea"))))
print(''.join(dropwhile(lambda c: c in "aeiou", reversed("idea")))[::-1])

['d', 'i']
id


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy đứng ở **cuối chuỗi** và lùi dần sang trái: nếu ký tự hiện tại là nguyên âm thì bỏ nó, tiếp tục lùi; ngay khi gặp **phụ âm** thì dừng. Những gì còn lại ở bên trái (kể cả nguyên âm trước đó) giữ nguyên.

Tính tay `"idea"` (các chỉ số `0 i`, `1 d`, `2 e`, `3 a`):

| Bước | vị trí | ký tự | nguyên âm? | hành động |
|---|---|---|---|---|
| 1 | 3 | `a` | có | bỏ, lùi |
| 2 | 2 | `e` | có | bỏ, lùi |
| 3 | 1 | `d` | không | dừng |

Giữ lại các vị trí `0..1` -> `"id"`.

Ví dụ khó hơn `"queueing"`: ký tự cuối `g` là phụ âm nên dừng ngay, kết quả giữ nguyên. Còn `"beautiful"`: cuối `l` -> giữ nguyên. `"quiet"`: cuối `t` -> giữ nguyên; `"banana"`: `a` bỏ, `n` dừng -> `"banan"`.

In [8]:
def explain(s):
    i = len(s) - 1
    while i >= 0 and s[i] in "aeiou":
        print(f"  bỏ s[{i}] = {s[i]!r}")
        i -= 1
    print(f"  dừng tại i={i}; giữ s[:{i + 1}] = {s[:i + 1]!r}")
    return s[:i + 1]

for s in ["idea", "banana", "aeiou", "day"]:
    print(repr(s)); explain(s)

'idea'
  bỏ s[3] = 'a'
  bỏ s[2] = 'e'
  dừng tại i=1; giữ s[:2] = 'id'
'banana'
  bỏ s[5] = 'a'
  dừng tại i=4; giữ s[:5] = 'banan'
'aeiou'
  bỏ s[4] = 'u'
  bỏ s[3] = 'o'
  bỏ s[2] = 'i'
  bỏ s[1] = 'e'
  bỏ s[0] = 'a'
  dừng tại i=-1; giữ s[:0] = ''
'day'
  dừng tại i=2; giữ s[:3] = 'day'


## 5. Cách 1 — Brute Force / baseline

Lặp: chừng nào chuỗi còn khác rỗng và ký tự cuối là nguyên âm thì cắt bỏ ký tự cuối bằng `s = s[:-1]`.

Nút thắt: mỗi lần `s[:-1]` tạo **một chuỗi mới** (sao chép `O(n)` ký tự). Xóa `k` nguyên âm cuối tốn `O(k · n)`; trường hợp xấu nhất `"aeiou..."` toàn nguyên âm là `O(n²)`. Với `n <= 100` không vấn đề, nhưng ta có thể cắt đúng một lần.

In [9]:
class SolutionBrute:
    def trimTrailingVowels(self, s):
        while s and s[-1] in "aeiou":
            s = s[:-1]                      # tạo chuỗi mới mỗi lần
        return s

print(repr(SolutionBrute().trimTrailingVowels("idea")), repr(SolutionBrute().trimTrailingVowels("day")), repr(SolutionBrute().trimTrailingVowels("aeiou")))

'id' 'day' ''


**Độ phức tạp:** Time `O(n²)` ở xấu nhất (mỗi lần cắt sao chép `O(n)`), Space `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần cắt từng ký tự một. Chỉ cần **tìm vị trí** `end` là chỉ số đầu tiên (tính từ phải) của một phụ âm, rồi cắt **một lần** `s[:end + 1]`. Con trỏ `i` chạy từ cuối về đầu và dừng ở phụ âm đầu tiên gặp phải.

**Invariant:** mọi ký tự ở vị trí `> i` đều là nguyên âm (thuộc đoạn cần xóa); ký tự tại `i` chưa xét. Vòng lặp kết thúc khi `i < 0` (toàn nguyên âm) hoặc `s[i]` là phụ âm, khi đó phần cần giữ là `s[:i + 1]`.

Quy tắc "nếu... thì...": nếu `s[i]` là nguyên âm thì `i -= 1`; ngược lại dừng. Các công cụ dựng sẵn (`rstrip`, `re.sub`, `dropwhile`) làm đúng việc này.

## 7. Các cách tiếp theo

### Cách 2 — Con trỏ từ cuối, cắt một lần

Dùng `i` lùi từ cuối, rồi `s[:i + 1]`. Chỉ một lần cắt nên tổng `O(n)`.

In [10]:
class SolutionPointer:
    def trimTrailingVowels(self, s):
        i = len(s) - 1
        while i >= 0 and s[i] in "aeiou":
            i -= 1
        return s[:i + 1]

print(repr(SolutionPointer().trimTrailingVowels("idea")), repr(SolutionPointer().trimTrailingVowels("day")), repr(SolutionPointer().trimTrailingVowels("aeiou")))

'id' 'day' ''


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (chuỗi kết quả).

### Cách 3 — `str.rstrip("aeiou")`

Một dòng. `rstrip` nhận **tập ký tự** nên đúng yêu cầu "mọi nguyên âm ở cuối".

In [11]:
class SolutionRstrip:
    def trimTrailingVowels(self, s):
        return s.rstrip("aeiou")

print(repr(SolutionRstrip().trimTrailingVowels("idea")), repr(SolutionRstrip().trimTrailingVowels("day")), repr(SolutionRstrip().trimTrailingVowels("aeiou")))

'id' 'day' ''


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### Cách 4 — Biểu thức chính quy

`re.sub(r"[aeiou]+$", "", s)`: xóa đoạn nguyên âm liên tiếp sát cuối. Dùng khi mẫu phức tạp hơn (ví dụ thêm điều kiện khác).

In [12]:
import re

class SolutionRegex:
    def trimTrailingVowels(self, s):
        return re.sub(r"[aeiou]+$", "", s)

print(repr(SolutionRegex().trimTrailingVowels("idea")), repr(SolutionRegex().trimTrailingVowels("day")), repr(SolutionRegex().trimTrailingVowels("aeiou")))

'id' 'day' ''


**Độ phức tạp:** Time `O(n)` (với mẫu đơn giản này), Space `O(n)`.

### Cách 5 — `dropwhile` trên chuỗi đảo ngược

Đảo chuỗi, bỏ nguyên âm ở đầu bằng `dropwhile`, nối lại và đảo ngược trở lại.

In [13]:
from itertools import dropwhile

class SolutionDropWhile:
    def trimTrailingVowels(self, s):
        kept = dropwhile(lambda c: c in "aeiou", reversed(s))
        return ''.join(kept)[::-1]

print(repr(SolutionDropWhile().trimTrailingVowels("idea")), repr(SolutionDropWhile().trimTrailingVowels("day")), repr(SolutionDropWhile().trimTrailingVowels("aeiou")))

'id' 'day' ''


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 với `"banana"` và `"aeiou"`:

| chuỗi | i bắt đầu | các bước | i dừng | kết quả |
|---|---|---|---|---|
| `"banana"` | 5 | `a` (bỏ) -> i=4, `n` dừng | 4 | `s[:5] = "banan"` |
| `"aeiou"` | 4 | `u,o,i,e,a` đều bỏ | -1 | `s[:0] = ""` |

In [14]:
def trace_pointer(s):
    i = len(s) - 1
    print(f"s={s!r}, bắt đầu i={i}")
    while i >= 0 and s[i] in "aeiou":
        print(f"  i={i}: s[i]={s[i]!r} là nguyên âm -> lùi")
        i -= 1
    if i >= 0:
        print(f"  i={i}: s[i]={s[i]!r} là phụ âm -> dừng")
    else:
        print("  i=-1: hết chuỗi")
    res = s[:i + 1]
    print(f"  kết quả s[:{i + 1}] = {res!r}")
    return res

assert trace_pointer("banana") == "banan"
assert trace_pointer("aeiou") == ""
assert trace_pointer("idea") == "id"
assert trace_pointer("day") == "day"

s='banana', bắt đầu i=5
  i=5: s[i]='a' là nguyên âm -> lùi
  i=4: s[i]='n' là phụ âm -> dừng
  kết quả s[:5] = 'banan'
s='aeiou', bắt đầu i=4
  i=4: s[i]='u' là nguyên âm -> lùi
  i=3: s[i]='o' là nguyên âm -> lùi
  i=2: s[i]='i' là nguyên âm -> lùi
  i=1: s[i]='e' là nguyên âm -> lùi
  i=0: s[i]='a' là nguyên âm -> lùi
  i=-1: hết chuỗi
  kết quả s[:0] = ''
s='idea', bắt đầu i=3
  i=3: s[i]='a' là nguyên âm -> lùi
  i=2: s[i]='e' là nguyên âm -> lùi
  i=1: s[i]='d' là phụ âm -> dừng
  kết quả s[:2] = 'id'
s='day', bắt đầu i=2
  i=2: s[i]='y' là phụ âm -> dừng
  kết quả s[:3] = 'day'


In [15]:
def trace_brute(s):
    step = 0
    while s and s[-1] in "aeiou":
        step += 1
        print(f"bước {step}: {s!r} -> {s[:-1]!r}")
        s = s[:-1]
    print("kết quả:", repr(s))
    return s

assert trace_brute("queueue") == "queueue".rstrip("aeiou")

bước 1: 'queueue' -> 'queueu'
bước 2: 'queueu' -> 'queue'
bước 3: 'queue' -> 'queu'
bước 4: 'queu' -> 'que'
bước 5: 'que' -> 'qu'
bước 6: 'qu' -> 'q'
kết quả: 'q'


## 9. Code nộp LeetCode

Dùng con trỏ từ cuối (Cách 2): rõ ràng, không phụ thuộc thư viện.

In [16]:
from typing import List, Optional


class Solution:
    def trimTrailingVowels(self, s: str) -> str:
        i = len(s) - 1
        while i >= 0 and s[i] in "aeiou":
            i -= 1
        return s[:i + 1]


print(repr(Solution().trimTrailingVowels("idea")), repr(Solution().trimTrailingVowels("day")), repr(Solution().trimTrailingVowels("aeiou")))

'id' 'day' ''


## 10. Vì sao thuật toán đúng

1. **Đoạn cần xóa là hậu tố liên tiếp.** Đề yêu cầu xóa các nguyên âm ở cuối; điều đó là hậu tố dài nhất gồm toàn nguyên âm.
2. **Con trỏ dừng đúng chỗ.** Vòng lặp lùi `i` chừng nào `s[i]` là nguyên âm. Khi dừng, hoặc `i = -1` (hết chuỗi) hoặc `s[i]` là phụ âm. Mọi ký tự sau `i` đều là nguyên âm (invariant), còn `s[i]` không phải nguyên âm nên hậu tố nguyên âm dài nhất bắt đầu ngay sau `i`.
3. **Cắt đúng.** `s[:i + 1]` giữ các ký tự `0..i`. Khi `i = -1` ta có `s[:0] = ""` đúng với trường hợp toàn nguyên âm.
4. **Không đụng nguyên âm ở nơi khác.** Ta chỉ xóa hậu tố, nên nguyên âm ở giữa/đầu (như `i` trong `"idea"`) được giữ nguyên.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

- Cách 1: `O(n²)` xấu nhất (mỗi lần `s[:-1]` sao chép), `O(n)` bộ nhớ.
- Cách 2, 3, 4, 5: `O(n)` thời gian; bộ nhớ `O(n)` cho chuỗi kết quả (chuỗi Python bất biến nên phải tạo chuỗi mới).
- Trường hợp tốt nhất: ký tự cuối là phụ âm -> chỉ kiểm tra một ký tự (nhưng vẫn phải tạo kết quả `O(n)` khi cắt).
- Trường hợp xấu nhất: toàn nguyên âm -> con trỏ đi hết chuỗi.

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên (toàn nguyên âm, toàn phụ âm, một ký tự) và 400 chuỗi ngẫu nhiên.

In [17]:
import random, string

solutions = [SolutionBrute(), SolutionPointer(), SolutionRstrip(), SolutionRegex(), SolutionDropWhile(), Solution()]

fixed = [
    ("idea", "id"),
    ("day", "day"),
    ("aeiou", ""),
    ("a", ""),
    ("b", "b"),
    ("banana", "banan"),
    ("aaab", "aaab"),             # nguyên âm ở đầu, cuối là phụ âm
    ("baaa", "b"),
    ("eat", "eat"),
    ("queue", "q"),
    ("sky", "sky"),               # y không phải nguyên âm
    ("xyzaeiou", "xyz"),
]
for s_, expected in fixed:
    for sol in solutions:
        assert sol.trimTrailingVowels(s_) == expected, (type(sol).__name__, s_)

random.seed(38560)
for _ in range(400):
    n = random.randint(1, 100)
    alphabet = random.choice(["aeiou", "abcde", string.ascii_lowercase, "aeiouy"])
    s_ = ''.join(random.choice(alphabet) for _ in range(n))
    outs = {sol.trimTrailingVowels(s_) for sol in solutions}
    assert len(outs) == 1, s_
    r = outs.pop()
    assert s_.startswith(r) and (r == "" or r[-1] not in "aeiou")
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Áp dụng hai lần cho cùng kết quả như áp dụng một lần (idempotent). (2) Thêm phụ âm vào cuối chuỗi thì chuỗi không bị cắt. (3) Độ dài kết quả bằng `n` trừ độ dài hậu tố nguyên âm.

In [18]:
random.seed(3856)
for _ in range(300):
    s_ = ''.join(random.choice("abcdeiouxyz") for _ in range(random.randint(1, 100)))
    r = Solution().trimTrailingVowels(s_)
    assert Solution().trimTrailingVowels(r) == r
    assert Solution().trimTrailingVowels(s_ + "z") == s_ + "z"
    tail = len(s_) - len(s_.rstrip("aeiou"))
    assert len(r) == len(s_) - tail
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: xóa mọi nguyên âm trong chuỗi

Dùng `replace` hoặc lọc toàn chuỗi xóa cả nguyên âm ở đầu/giữa. Với `"idea"` nhận `"d"` thay vì `"id"`.

In [19]:
def wrong_remove_all(s):
    return ''.join(c for c in s if c not in "aeiou")          # SAI: xóa cả nguyên âm giữa/đầu

print(repr(wrong_remove_all("idea")), "!=", repr(Solution().trimTrailingVowels("idea")))
assert wrong_remove_all("idea") == "d" and Solution().trimTrailingVowels("idea") == "id"

'd' != 'id'


### Lỗi 2: chỉ xóa một nguyên âm cuối (dùng `if` thay vì `while`)

Đề yêu cầu xóa **tất cả** nguyên âm liên tiếp ở cuối. `if` chỉ xóa một ký tự.

In [20]:
def wrong_if(s):
    if s and s[-1] in "aeiou":                                # SAI: phải là while
        s = s[:-1]
    return s

print(repr(wrong_if("idea")), "!=", repr(Solution().trimTrailingVowels("idea")))
assert wrong_if("idea") == "ide" and Solution().trimTrailingVowels("idea") == "id"

'ide' != 'id'


### Lỗi 3: dùng `strip` (hai đầu) hoặc `lstrip` thay vì `rstrip`

`strip("aeiou")` còn xóa nguyên âm ở **đầu** chuỗi: `"idea".strip("aeiou")` ra `"d"`.

In [21]:
def wrong_strip(s):
    return s.strip("aeiou")                                   # SAI: xóa cả đầu

def wrong_lstrip(s):
    return s.lstrip("aeiou")                                  # SAI: xóa ở đầu thay vì cuối

print(repr(wrong_strip("idea")), repr(wrong_lstrip("idea")), repr(Solution().trimTrailingVowels("idea")))
assert wrong_strip("idea") == "d" and wrong_lstrip("idea") == "dea" and Solution().trimTrailingVowels("idea") == "id"

'd' 'dea' 'id'


### Lỗi 4: nhầm `removesuffix("aeiou")` với `rstrip("aeiou")`

`removesuffix` xóa **một hậu tố đúng bằng chuỗi `"aeiou"`**, không xóa từng ký tự thuộc tập. `"idea".removesuffix("aeiou")` vẫn là `"idea"`.

In [22]:
def wrong_removesuffix(s):
    return s.removesuffix("aeiou")                            # SAI: xem "aeiou" là một chuỗi con

print(repr(wrong_removesuffix("idea")), "!=", repr(Solution().trimTrailingVowels("idea")))
assert wrong_removesuffix("idea") == "idea" and wrong_removesuffix("aeiou") == ""
assert Solution().trimTrailingVowels("idea") == "id"

'idea' != 'id'


### Lỗi 5: lỗi lệch một đơn vị khi cắt

Cắt `s[:i]` thay vì `s[:i + 1]` mất luôn phụ âm cuối cùng của phần được giữ lại.

In [23]:
def wrong_off_by_one(s):
    i = len(s) - 1
    while i >= 0 and s[i] in "aeiou":
        i -= 1
    return s[:i]                                               # SAI: phải s[:i + 1]

print(repr(wrong_off_by_one("idea")), "!=", repr(Solution().trimTrailingVowels("idea")))
assert wrong_off_by_one("idea") == "i" and wrong_off_by_one("day") == "da"
assert Solution().trimTrailingVowels("day") == "day"

'i' != 'id'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xóa ... ở **cuối**/ở **đầu**" -> con trỏ từ đầu/cuối rồi cắt lát một lần, hoặc `rstrip`/`lstrip`.
- `rstrip(chars)` nhận **tập ký tự**; muốn xóa hậu tố cố định dùng `removesuffix` (Python 3.9+).
- Nhớ định nghĩa nguyên âm của đề (ở đây **không** gồm `y`).
- Chuỗi bất biến: tránh `s = s[:-1]` lặp nhiều lần với chuỗi dài; hãy tìm vị trí rồi cắt một lần.
- Test cơ bản: toàn nguyên âm, toàn phụ âm, nguyên âm xen kẽ, một ký tự.

## 15. Biến thể của bài

1. **Remove Vowels from a String** (1119): xóa mọi nguyên âm.
2. **Reverse Vowels of a String** (345): đảo các nguyên âm.
3. **Trim trailing zeros** / **Remove Trailing Zeros From a String** (2710): cắt hậu tố các chữ số `0`.
4. **Remove Trailing Whitespace / Strip**: tương tự với khoảng trắng.
5. **Longest Substring Of All Vowels in Order** (1839): làm việc với đoạn nguyên âm.
6. Biến thể: xóa nguyên âm cả ở **đầu và cuối** (`strip`), hoặc xóa nguyên âm cuối cho **từng từ** trong câu.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cắt từng ký tự | `while ... s = s[:-1]` | `O(n²)` | `O(n)` | Ý tưởng thẳng nhất, `n` nhỏ |
| 2. Con trỏ | lùi `i`, cắt một lần | `O(n)` | `O(n)` | Nộp bài, rõ ràng |
| 3. `rstrip` | thư viện chuẩn | `O(n)` | `O(n)` | Ngắn gọn nhất |
| 4. `re.sub` | mẫu `[aeiou]+$` | `O(n)` | `O(n)` | Khi điều kiện phức tạp hơn |
| 5. `dropwhile` | bỏ đầu của chuỗi đảo | `O(n)` | `O(n)` | Phong cách iterator |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().trimTrailingVowels("education")` và `Solution().trimTrailingVowels("queue")` ra gì?

**Bài 2 (tính tay):** `"aaaabcaa"` sau khi bỏ nguyên âm cuối còn gì?

**Bài 3 (điền TODO):** hoàn thành `count_trailing_vowels(s)` trả về **số** nguyên âm ở cuối chuỗi.

**Bài 4 (tự viết lại):** viết `trim_vowels_both_ends(s)` xóa nguyên âm liên tiếp ở **cả đầu và cuối** chuỗi.

**Bài 5 (mini challenge):** viết `trim_trailing_vowels_each_word(sentence)` với `sentence` gồm các từ cách nhau bởi một dấu cách; xóa nguyên âm cuối của **từng từ** (từ toàn nguyên âm thành rỗng và vẫn giữ dấu cách tương ứng).

In [24]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(repr(Solution().trimTrailingVowels("education")), repr(Solution().trimTrailingVowels("queue")))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_trailing_vowels_todo(s):
    count = 0
    # TODO: lùi từ cuối, đếm số nguyên âm liên tiếp
    return count

'education' 'q'


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `"education"` kết thúc bằng `n`; `"queue"` kết thúc bằng `ueue`. Bài 2: hai ký tự cuối `aa` là nguyên âm.

**Mức 2 (vừa):** Bài 3: dùng `len(s) - len(s.rstrip("aeiou"))` hoặc vòng lặp lùi. Bài 4: `strip("aeiou")`. Bài 5: `' '.join(w.rstrip('aeiou') for w in sentence.split(' '))` (dùng `split(' ')` để giữ các từ rỗng).

**Mức 3 (gần đáp án):** Bài 1: `"education"` và `"q"`. Bài 2: `"aaaabc"`. Bài 5: `"idea day"` -> `"id day"`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().trimTrailingVowels("education") == "education"
assert Solution().trimTrailingVowels("queue") == "q"
assert Solution().trimTrailingVowels("aaaabcaa") == "aaaabc"

# Bài 3
def count_trailing_vowels(s):
    i = len(s) - 1
    count = 0
    while i >= 0 and s[i] in "aeiou":
        count += 1
        i -= 1
    return count
assert count_trailing_vowels("idea") == 2 and count_trailing_vowels("aeiou") == 5 and count_trailing_vowels("day") == 0

# Bài 4
def trim_vowels_both_ends(s):
    return s.strip("aeiou")
assert trim_vowels_both_ends("idea") == "d" and trim_vowels_both_ends("aeiou") == "" and trim_vowels_both_ends("banana") == "banan"

# Bài 5
def trim_trailing_vowels_each_word(sentence):
    return ' '.join(Solution().trimTrailingVowels(w) for w in sentence.split(' '))
assert trim_trailing_vowels_each_word("idea day") == "id day"
assert trim_trailing_vowels_each_word("aeiou x") == " x"
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "bỏ hậu tố thỏa điều kiện":**

```python
i = len(s) - 1
while i >= 0 and s[i] in "aeiou":   # điều kiện bỏ
    i -= 1
return s[:i + 1]
# hoặc ngắn gọn: s.rstrip("aeiou")
```

**Checklist trước khi submit:**

- [ ] Chỉ xóa **hậu tố** các nguyên âm liên tiếp ở cuối (không đụng nguyên âm khác)?
- [ ] Dùng `while` (xóa hết), không phải `if`?
- [ ] Điều kiện biên `i >= 0` đặt **trước** `s[i]` để không lỗi chỉ số?
- [ ] Cắt `s[:i + 1]`, không phải `s[:i]`?
- [ ] `y` không phải nguyên âm; đã thử `"day"`, `"idea"`, `"aeiou"`?
- [ ] Chữ ký đúng `trimTrailingVowels(self, s: str) -> str`?